In [ ]:
import pandas as pd
import numpy as np
import random 
from grid_search import estimate_single_config

from joblib import Parallel, delayed
from tqdm.auto import tqdm

import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

import plot_style

plot_style.apply()
random.seed(42)

c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
names = pd.read_csv("C:\\Users\\jonat\\Downloads\\kq36gn36yooczs1o_csv\\kq36gn36yooczs1o.csv")

In [2]:
#import the wide dataframe
df = pd.read_csv('../../data/X.csv', index_col=0)

#import the features dataframe
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)

In [3]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [4]:
#remove columns that are not just numbers in the name
df = df[[col for col in df.columns if col.replace('.','',1).isdigit()]]

# transform the df data into log returns from normal returns
df = np.log(1 + df)

#convert index to datetime
df.index = pd.to_datetime(df.index)

In [5]:
# delete te frist row of df
df = df.iloc[1:]

In [6]:
# tryng to run the estimation on one stock first
y = df['10145']
X = feature_matrix.copy()

#change all the ' ' in the column names to '_'
X.columns = [col.replace(' ', '_') for col in X.columns]

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each (or use len(...) for "all")
num_topics = len(topic_cols)  # e.g., 100
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# # usage
X = to_ar1_innovations(X)

# remove first row in X
X = X.iloc[1:]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

#convert y index to datetime
#y.index = pd.to_datetime(y.index)
# ensure that all indices align

common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [7]:
def estimate_multiple_targets(X, y_df, window_size, n_lags, lambda_val, 
                               standardize=True, verbose=False, 
                               save_only_positive_r_squared=True, 
                               return_details=True,
                               raise_errors=False):
    """
    Run estimate_single_config for multiple target columns.
    
    Parameters
    ----------
    X : pd.DataFrame or np.ndarray
        Feature matrix (same for all targets)
    y_df : pd.DataFrame or pd.Series
        Target variable(s). If DataFrame, each column is a separate target.
        If Series, treated as single target.
    window_size : int
        Rolling window size
    n_lags : int
        Number of lags
    lambda_val : float
        Regularization parameter
    standardize : bool, default=True
        Whether to standardize features
    verbose : bool, default=False
        Whether to print progress
    save_only_positive_r_squared : bool, default=True
        Whether to save only positive R² results
    return_details : bool, default=True
        Whether to return detailed coefficients
    raise_errors : bool, default=False
        If True, raise errors instead of catching them
    
    Returns
    -------
    dict with keys for each target column:
        - Each key contains {'summary': dict, 'details': DataFrame}
    """
    # Handle single Series input
    if isinstance(y_df, pd.Series):
        y_df = y_df.to_frame()
    
    # Convert to DataFrame if numpy array
    if isinstance(y_df, np.ndarray):
        if y_df.ndim == 1:
            y_df = pd.DataFrame({'target': y_df})
        else:
            y_df = pd.DataFrame(y_df, columns=[f'target_{i}' for i in range(y_df.shape[1])])
    
    results = {}
    target_cols = y_df.columns
    
    if verbose:
        print(f"Estimating for {len(target_cols)} target column(s)...")
        print(f"X shape: {X.shape if hasattr(X, 'shape') else 'unknown'}")
        print(f"y_df shape: {y_df.shape}")
    
    for i, col in enumerate(target_cols):
        if verbose:
            print(f"  Processing column {i+1}/{len(target_cols)}: {col}")
        
        y_single = y_df[col]
        
        # Check for issues with the target column
        if verbose:
            print(f"    y_single shape: {y_single.shape}")
            print(f"    y_single non-null: {y_single.notna().sum()}/{len(y_single)}")
            print(f"    y_single dtype: {y_single.dtype}")
        
        try:
            result = estimate_single_config(
                X=X,
                y=y_single,
                window_size=window_size,
                n_lags=n_lags,
                lambda_val=lambda_val,
                standardize=standardize,
                verbose=False,  # Individual runs stay quiet
                save_only_positve_r_squared=save_only_positive_r_squared,
                return_details=return_details
            )
            
            # Add target column name to summary
            result['summary']['target_column'] = col
            
            # Add target column name to details if available
            if not result['details'].empty:
                result['details']['target_column'] = col
            
            results[col] = result
            
            if verbose:
                r2_oos = result['summary'].get('r2_oos_stage2', np.nan)
                error_msg = result['summary'].get('error', None)
                if error_msg:
                    print(f"    ✗ Error captured: {error_msg}")
                else:
                    print(f"    ✓ R² OOS Stage 2: {r2_oos:.4f}" if not np.isnan(r2_oos) else "    ✓ Complete")
        
        except Exception as e:
            if raise_errors:
                raise
            
            if verbose:
                print(f"    ✗ Error: {str(e)}")
                import traceback
                print(f"    Traceback: {traceback.format_exc()}")
            
            results[col] = {
                'summary': {
                    'target_column': col,
                    'window_size': window_size,
                    'n_lags': n_lags,
                    'lambda': lambda_val,
                    'error': str(e),
                    'kappa': np.nan,
                    'r2_oos_stage2': np.nan
                },
                'details': pd.DataFrame()
            }
    
    return results


# Diagnostic function to check what went wrong
def diagnose_errors(results):
    """
    Print all errors from the results dictionary.
    
    Parameters
    ----------
    results : dict
        Output from estimate_multiple_targets
    """
    print("Error Diagnosis:")
    print("=" * 60)
    
    for target_name, result in results.items():
        error = result['summary'].get('error', None)
        if error:
            print(f"\nTarget: {target_name}")
            print(f"Error: {error}")
    
    # Count errors
    n_errors = sum(1 for r in results.values() if 'error' in r['summary'])
    n_success = len(results) - n_errors
    print(f"\n{'=' * 60}")
    print(f"Summary: {n_success} successful, {n_errors} errors out of {len(results)} total")


def aggregate_multiple_target_results(results, metric='r2_oos_stage2'):
    """
    Aggregate results across multiple targets.
    
    Parameters
    ----------
    results : dict
        Output from estimate_multiple_targets
    metric : str, default='r2_oos_stage2'
        Metric to extract from summaries
    
    Returns
    -------
    pd.DataFrame
        Summary table with one row per target
    """
    summary_list = []
    
    for target_name, result in results.items():
        summary_list.append(result['summary'])
    
    return pd.DataFrame(summary_list)


def combine_details_across_targets(results):
    """
    Combine all detail DataFrames into a single DataFrame.
    
    Parameters
    ----------
    results : dict
        Output from estimate_multiple_targets
    
    Returns
    -------
    pd.DataFrame
        Combined details with target_column identifier
    """
    all_details = []
    
    for target_name, result in results.items():
        if not result['details'].empty:
            all_details.append(result['details'])
    
    if all_details:
        return pd.concat(all_details, ignore_index=True)
    else:
        return pd.DataFrame()


In [8]:
# create subsample of df of 2 stocks for testing randomly
subsample_stocks = random.sample([col for col in df.columns if col.replace('.','',1).isdigit()], 2)
df = df[subsample_stocks]

In [9]:
df.columns

Index(['89138', '25304'], dtype='object')

In [11]:
# results = estimate_multiple_targets(
#     X=X,
#     y_df=df,  # DataFrame with multiple columns
#     window_size=120,
#     n_lags=3,
#     lambda_val=0.01,
#     verbose=True,  # Enable verbose
#     raise_errors=False
# )

# Access results for a specific target
target1_summary = results['89138']['summary']
target1_details = results['89138']['details']

# Get summary table across all targets
summary_table = aggregate_multiple_target_results(results)
print(summary_table[['target_column', 'r2_oos_stage2', 'kappa', 'kappa_tstat']])

# Combine all details into one DataFrame
all_details = combine_details_across_targets(results)

  target_column  r2_oos_stage2         kappa   kappa_tstat
0         89138      -0.000681  3.816899e-01  4.527417e-01
1         25304      -4.681852  3.261676e-09  2.352274e-08


C:\Users\jonat\AppData\Local\Temp\ipykernel_38528\437955577.py:195: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_details, ignore_index=True)
